In [0]:
%run ../setup

In [0]:
spark.sql("USE SCHEMA silver_iter;")

In [0]:
spark.sql("SELECT * FROM bronze_iter.iter_delta_raw limit 20;").display()

In [0]:
spark.sql("""
          SELECT
          ENTIDAD as ent, 
          * 
          FROM bronze_iter.iter_delta_raw 
          WHERE ENTIDAD = 20 AND MUN = 73 AND LOC !=0
          limit 200;""").display()

Databricks visualization. Run in Databricks to view.

In [0]:
spark.sql("""
          SELECT
          ENTIDAD as ent, 
          * 
          FROM bronze_iter.iter_delta_raw 
          WHERE ENTIDAD = 20 AND MUN = 469 AND LOC !=0
          ORDER BY POBTOT DESC
          limit 200;""").display()

Databricks data profile. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
spark.sql(f"""
        CREATE OR REPLACE TABLE iter_delta_localidad USING delta LOCATION 's3://{buckett}/silver_i/iter_delta_localidad'
        WITH estado AS (
            SELECT
            DISTINCT loc,
            --nom_ent,
            entidad,
            mun,
            --nom_mun,
            --loc,
            --*
	          cast(CONCAT(ENTIDAD, '_', MUN, '_', LOC) AS varchar(128)) AS key_localidad,
            nom_loc,
            longitud,
            latitud, 
            altitud
            FROM bronze_iter.iter_delta_raw 
            WHERE entidad != 0 AND mun != 0 AND loc !=0
            --ORDER BY entidad, mun, loc
        )
          SELECT
          ROW_NUMBER() OVER(ORDER BY entidad, mun, loc) AS id_localidad,
          --mun,
          key_localidad,
          nom_loc,
          longitud,
          latitud,
          altitud
          --* 
          FROM estado
          ;""").display()

In [0]:
#spark.sql(f"DROP table iter_delta_localidad")

In [0]:
#spark.sql(f"DROP TABLE delta.`s3://{buckett}/silver_i/iter_delta_localidad`")

In [0]:
spark.sql(f"SELECT * FROM delta.`s3://{buckett}/silver_i/iter_delta_localidad`").display()

In [0]:
spark.sql(f"DESCRIBE history delta.`s3://{buckett}/silver_i/iter_delta_localidad`").display()

In [0]:
spark.sql("DESCRIBE EXTENDED bronze_iter.iter_delta_raw").display()

In [0]:
spark.sql("SELECT count(*) FROM bronze_iter.iter_delta_raw").display()

In [0]:
spark.sql(f"""
        --CREATE OR REPLACE TABLE iter_delta_municipio USING delta LOCATION 's3://{buckett}/silver_i/iter_delta_municipio'
        WITH estado AS (
            SELECT
            DISTINCT mun as mun_d,
            --nom_ent,
            entidad,
            mun,
            nom_mun,
            --loc,
            --*
	        cast(CONCAT(ENTIDAD, '_', MUN) AS varchar(128)) AS key_municipio,
            nom_mun As nom_municipio
            FROM bronze_iter.iter_delta_raw 
            WHERE entidad != 0 AND mun != 0
            --ORDER BY entidad, mun, loc
        )
          SELECT
          ROW_NUMBER() OVER(ORDER BY entidad, mun_d) AS id_municipio,
          --nom_ent,
          --entidad,
          --mun,
          key_municipio,
          nom_municipio
          --* 
          FROM estado
          ;""").display()

In [0]:
spark.sql(f"""
        CREATE OR REPLACE TABLE iter_delta_estado USING delta LOCATION 's3://{buckett}/silver_i/iter_delta_estado'
        WITH estado AS (
            SELECT
            DISTINCT entidad as ent_id,
            nom_ent,
            entidad
            --mun,
            --nom_mun
            --loc,
            --*
	        --cast(CONCAT(ENTIDAD, '_', MUN) AS varchar(128)) AS key_municipio,
            --nom_mun As nom_municipio
            FROM bronze_iter.iter_delta_raw 
            WHERE entidad != 0 
            --ORDER BY entidad, mun, loc
        )
          SELECT
          ROW_NUMBER() OVER(ORDER BY entidad) AS id_municipio,
          nom_ent
          --entidad
          --mun,
          --key_municipio,
          --nom_municipio
          --* 
          FROM estado
          ;""").display()

In [0]:
spark.sql(f"SELECT * FROM delta.`s3://{buckett}/silver_i/iter_delta_estado`").display()

In [0]:
spark.sql(f"""
        --CREATE OR REPLACE TABLE iter_delta_localidad USING delta LOCATION 's3://{buckett}/silver_i/iter_delta_localidad'
        WITH estado AS (
            SELECT
            --DISTINCT loc,
            --nom_ent,
            entidad as id_entidad,
            --mun,
            --nom_mun,
            --loc,
            --*
            cast(CONCAT(ENTIDAD, '_', MUN) AS varchar(128)) AS key_municipio,
	          cast(CONCAT(ENTIDAD, '_', MUN, '_', LOC) AS varchar(128)) AS key_localidad,
            --nom_loc
            *
            FROM bronze_iter.iter_delta_raw 
            WHERE entidad != 0 AND mun != 0 AND loc !=0
            --ORDER BY entidad, mun, loc
            --limit 100
        )
          SELECT
          ROW_NUMBER() OVER(ORDER BY entidad, mun, loc) AS id_mex,
          --mun,
          --key_localidad,
          --nom_loc
          * 
          FROM estado as e
          --LEFT JOIN iter_delta_localidad AS l ON e.key_localidad = l.key_localidad
          --LEFT JOIN iter_delta_municipio AS m ON e.key_municipio = m.key_municipio
          ;""").display()

In [0]:
df = spark.read.option("inferSchema", "true").csv("/ruta", header=True)